# 尺寸重要吗——训练尺寸实验

Ed 提到 OpenAI 建议使用 50-100 个示例进行微调，但是当您改变训练集大小时实际会发生什么？

该笔记本使用 50、100、200、400、1000、2000 和 5000 个训练示例对 GPT-4.1-nano 进行微调，然后绘制准确性的变化情况。所有 7 个作业在 OpenAI 上并行运行，与一项作业相同的等待。

**数据集：** `ed-donner/items_lite` （20K 亚马逊产品，带摘要 + 价格，0.50-999 美元）

In [ ]:
# 进口

import os
import re
import io
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI
from tqdm.notebook import tqdm

In [ ]:
# 设置

load_dotenv(override=True)
client = OpenAI()

ds = load_dataset("ed-donner/items_lite")
train = ds["train"]
test = ds["test"]

print(f"Train: {len(train):,} items | Test: {len(test):,} items")
print(f"\nSample item:")
print(f"  Title: {train[0]['title']}")
print(f"  Price: ${train[0]['price']:.2f}")
print(f"  Summary: {train[0]['summary'][:200]}...")

## 评估助手

In [ ]:
# 后处理：从 LLM 输出中提取数字价格

def post_process(value):
    if isinstance(value, str):
        value = value.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", value)
        if not match:
            return 0
        price = float(match.group())
    else:
        price = float(value)
    
    # 剪辑至实际范围
    price = max(1.0, min(999.0, price))
    
    # 四舍五入到最接近的常见零售结尾 (.99, .95, .00)
    endings = [0.00, 0.49, 0.95, 0.99]
    base = int(price)
    best = min(endings, key=lambda e: abs(price - (base + e)))
    return round(base + best, 2)

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"

def color_for(error, truth):
    if error < 40 or error / truth < 0.2:
        return GREEN
    elif error < 80 or error / truth < 0.4:
        return YELLOW
    return RED


def evaluate(predictor, dataset, n=200):
    """Run predictor on n test items. Returns (predictions, actuals, mae)."""
    preds, actuals = [], []
    for i in tqdm(range(n)):
        item = dataset[i]
        guess = post_process(predictor(item))
        truth = item["price"]
        error = abs(guess - truth)
        c = color_for(error, truth)
        print(f"{c}${error:.0f}{RESET} ", end="")
        preds.append(guess)
        actuals.append(truth)
    mae = np.mean(np.abs(np.array(preds) - np.array(actuals)))
    print(f"\n\nMAE: ${mae:.2f}")
    return preds, actuals, mae

## 基线

In [ ]:
# 随机基线：猜测 1 美元到 999 美元之间的随机价格

random.seed(42)

def random_predictor(item):
    return random.randrange(1, 1000)

_, _, random_mae = evaluate(random_predictor, test)

In [ ]:
# 平均基线：始终预测训练集平均值

train_avg = np.mean([item["price"] for item in train])
print(f"Training average price: ${train_avg:.2f}")

def mean_predictor(item):
    return train_avg

_, _, mean_mae = evaluate(mean_predictor, test)

## 零样本 GPT-4.1-nano

这是我们的学习曲线的“0 个训练示例”数据点。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def zero_shot(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0
    )
    return response.choices[0].message.content

zero_preds, zero_actuals, zero_mae = evaluate(zero_shot, test)

## 准备并启动所有 7 个微调作业

训练规模：50、100、200、400、1000、2000、5000。全部都有相同的 50 个验证示例。

In [ ]:
SIZES = [50, 100, 200, 400, 1000, 2000, 5000]

def messages_for(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": f"${item['price']:.2f}"}
    ]

def make_jsonl(dataset, n):
    lines = []
    for i in range(n):
        row = json.dumps({"messages": messages_for(dataset[i])})
        lines.append(row)
    return "\n".join(lines)

# 构建验证 JSONL（所有作业都有相同的 50 个项目）
val = ds["validation"]
val_jsonl = make_jsonl(val, 50)

print(f"Validation set: 50 items")
print(f"Sample line: {json.loads(val_jsonl.split(chr(10))[0])['messages'][0]['content'][:80]}...")

In [ ]:
# 上传所有文件并启动所有作业（如果速率受限则重试）

import time

jobs = {}

# 上传共享验证文件一次
val_file = client.files.create(
    file=("val.jsonl", io.BytesIO(val_jsonl.encode())),
    purpose="fine-tune"
)
print(f"Uploaded validation file: {val_file.id}")

for size in SIZES:
    train_jsonl = make_jsonl(train, size)
    train_file = client.files.create(
        file=(f"train_{size}.jsonl", io.BytesIO(train_jsonl.encode())),
        purpose="fine-tune"
    )
    # 重试循环：OpenAI 允许最多 6 个并发作业
    while True:
        try:
            job = client.fine_tuning.jobs.create(
                training_file=train_file.id,
                validation_file=val_file.id,
                model="gpt-4.1-nano-2025-04-14",
                seed=42,
                hyperparameters={"n_epochs": 1, "batch_size": 1},
                suffix=f"pricer-{size}"
            )
            break
        except Exception as e:
            if "rate" in str(e).lower():
                print(f"  Size {size}: rate-limited, waiting 60s for a slot...")
                time.sleep(60)
            else:
                raise
    jobs[size] = job.id
    print(f"  Size {size}: job {job.id} launched")

print(f"\nAll {len(jobs)} jobs launched.")

## 监控作业

重新运行此单元以检查状态。一切应该大约在同一时间完成。

In [ ]:
# 检查所有作业的状态（重新运行直到所有作业显示“成功”）

all_done = True
for size, job_id in jobs.items():
    status = client.fine_tuning.jobs.retrieve(job_id)
    model_name = status.fine_tuned_model or "(training...)"
    print(f"  Size {size}: {status.status} -> {model_name}")
    if status.status != "succeeded":
        all_done = False

if all_done:
    print("\nAll jobs complete!")
else:
    print("\nStill training... re-run this cell in a few minutes.")

## 评估所有微调模型

In [ ]:
# 获取微调后的模型名称

ft_models = {}
for size, job_id in jobs.items():
    info = client.fine_tuning.jobs.retrieve(job_id)
    ft_models[size] = info.fine_tuned_model
    print(f"  Size {size}: {info.fine_tuned_model}")

In [ ]:
# 在相同的 200 个测试项目上评估每个微调模型

ft_results = {}

for size in SIZES:
    model_name = ft_models[size]
    print(f"\n--- Fine-tuned on {size} examples ({model_name}) ---")

    def ft_predictor(item, _model=model_name):
        prompt = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item['summary']}"
        response = client.chat.completions.create(
            model=_model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=10,
            temperature=0
        )
        return response.choices[0].message.content

    preds, actuals, mae = evaluate(ft_predictor, test)
    ft_results[size] = {"preds": preds, "actuals": actuals, "mae": mae}

＃＃ 结果

In [ ]:
# 汇总表

print(f"{'Model':<25} {'MAE':>10}")
print("-" * 37)
print(f"{'Random baseline':<25} ${random_mae:>8.2f}")
print(f"{'Mean baseline':<25} ${mean_mae:>8.2f}")
print(f"{'Zero-shot GPT-4.1-nano':<25} ${zero_mae:>8.2f}")
for size in SIZES:
    print(f"{'Fine-tuned (' + str(size) + ' ex.)':<25} ${ft_results[size]['mae']:>8.2f}")

### 学习曲线

In [ ]:
# 学习曲线：MAE 与训练规模

sizes_with_zero = [0] + SIZES
maes_curve = [zero_mae] + [ft_results[s]["mae"] for s in SIZES]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sizes_with_zero, maes_curve, "o-", color="#2563eb", linewidth=2, markersize=8)
ax.axhline(y=zero_mae, color="#f59e0b", linestyle="--", linewidth=1, label=f"Zero-shot baseline (${zero_mae:.0f})")

for x, y in zip(sizes_with_zero, maes_curve):
    offset = 12 if y >= zero_mae else -18
    ax.annotate(f"${y:.0f}", (x, y), textcoords="offset points",
                xytext=(0, offset), ha="center", fontsize=10)

ax.set_xlabel("Training examples")
ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("Learning Curve: How Much Training Data Does GPT-4.1-nano Need?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 所有方法的比较

In [ ]:
# 条形图：所有方法并排

labels = ["Random", "Mean", "Zero-shot"] + [f"FT-{s}" for s in SIZES]
values = [random_mae, mean_mae, zero_mae] + [ft_results[s]["mae"] for s in SIZES]
colors = ["#94a3b8", "#94a3b8", "#f59e0b"] + [
    "#10b981" if ft_results[s]["mae"] < zero_mae else "#2563eb" for s in SIZES
]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors)
ax.axhline(y=zero_mae, color="#f59e0b", linestyle="--", linewidth=1, alpha=0.7)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"${val:.0f}", ha="center", fontsize=9)

ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("All Approaches Compared (green = beats zero-shot)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 预测价格与实际价格

In [ ]:
# 散点图：零样本与最佳微调模型

best_size = min(ft_results, key=lambda s: ft_results[s]["mae"])
best = ft_results[best_size]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

max_price = max(max(zero_actuals), max(zero_preds),
                max(best["actuals"]), max(best["preds"]))

for ax, preds, actuals, title in [
    (ax1, zero_preds, zero_actuals, f"Zero-shot (MAE=${zero_mae:.0f})"),
    (ax2, best["preds"], best["actuals"], f"Fine-tuned {best_size} ex. (MAE=${best['mae']:.0f})"),
]:
    ax.scatter(actuals, preds, alpha=0.4, s=15, color="#2563eb")
    ax.plot([0, max_price], [0, max_price], "--", color="#ef4444", linewidth=1)
    ax.set_xlabel("Actual price ($)")
    ax.set_ylabel("Predicted price ($)")
    ax.set_title(title)
    ax.set_xlim(0, max_price)
    ax.set_ylim(0, max_price)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()